# Photo-quality energy: can it tell a retakeable photo from a good one?

**Cost: $0 in API credits** (no paid calls in this notebook). **Meta model:** DINOv2 (`facebook/dinov2-base`,
Meta's open-weights self-supervised encoder), frozen. Any GPU runtime works (T4 is enough; photos are streamed in small batches).

- **Good photos:** IDRiD (your Drive zips), assumed diagnostic quality.
- **Bad photos:** synthetic corruptions of those, in four groups a coach could name: focus, exposure, obstruction, aim.
- **Test:** DeepDRiD's *real* quality labels (1,600 photos, another hospital and camera; CC BY-SA 4.0),
  https://github.com/deepdrdoc/DeepDRiD. "Overall quality = 0" means not good enough for diagnosis.

Energies compared (all trained **without** DeepDRiD labels):
1. **Contrastive** (LeCun 2006): one energy per failure group, pushed down on good photos and up on that group's corruptions.
2. **Regularized** (LeCun 2022): model only good photos (Mahalanobis energy), no negatives.
3. **Simple checks:** sharpness, brightness, contrast. An energy model has to beat these to be worth anything.

A supervised reference (trained on DeepDRiD's own training labels) shows how far zero-shot is from the ceiling.

In [ ]:
import os, glob
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/ebl_fundus'
os.makedirs(f'{D}/quality', exist_ok=True)

!unzip -q -o "{D}/A. Segmentation.zip" -d /content/idrid
!unzip -q -o "{D}/B. Disease Grading.zip" -d /content/idrid
# DeepDRiD: fetch only the 1,600 regular fundus photos and their labels
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/deepdrdoc/DeepDRiD.git /content/deepdrid
!cd /content/deepdrid && git sparse-checkout set regular_fundus_images/regular-fundus-training regular_fundus_images/regular-fundus-validation && git checkout -q
print(len(glob.glob('/content/idrid/**/*.jpg', recursive=True)), 'IDRiD photos |',
      len(glob.glob('/content/deepdrid/regular_fundus_images/*/Images/*/*.jpg')), 'DeepDRiD photos')

Every photo is cropped to the retina, padded to a square and resized to 448 px. Corruptions are applied after that,
so they look like what a camera does, not like a framing difference between datasets.

In [ ]:
import numpy as np, torch, pandas as pd
from PIL import Image, ImageFilter
from scipy import ndimage
from concurrent.futures import ThreadPoolExecutor
S = 448
rng = np.random.default_rng(0)

def load_square(path):
    img = Image.open(path).convert('RGB'); img.thumbnail((1024, 1024))
    a = np.asarray(img); ys, xs = np.nonzero(a.max(-1) > 15)
    a = a[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = a.shape[:2]; side = max(h, w)
    sq = np.zeros((side, side, 3), np.uint8); sq[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = a
    return np.asarray(Image.fromarray(sq).resize((S, S), Image.BILINEAR)).astype(np.float32) / 255

def retina_mask(x):
    return x.max(-1) > 15 / 255

def blur(x, r):
    return np.asarray(Image.fromarray((x * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(r))).astype(np.float32) / 255

def corrupt(x, kind):
    m = retina_mask(x)[..., None]; yy, xx = np.mgrid[:S, :S] / S
    if kind == 'blur':      y = blur(x, rng.uniform(2, 5))
    elif kind == 'haze':    y = blur(x.mean((0, 1)) + (x - x.mean((0, 1))) * rng.uniform(0.35, 0.6), 1)
    elif kind == 'dark':    y = x * rng.uniform(0.3, 0.55)
    elif kind == 'washout': a = rng.uniform(0.35, 0.6); y = x * (1 - a) + 0.9 * a
    elif kind == 'shadow':  t = rng.uniform(0.2, 0.45); y = x * np.clip((yy - t) / 0.08, 0.15, 1)[..., None]
    elif kind == 'glare':
        y = x.copy()
        for _ in range(rng.integers(1, 4)):
            ang, r = rng.uniform(0, 2 * np.pi), rng.uniform(0.05, 0.15)
            cy, cx = 0.5 + 0.38 * np.sin(ang), 0.5 + 0.38 * np.cos(ang)
            y = y + rng.uniform(0.5, 0.9) * np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * r ** 2))[..., None]
    elif kind == 'aim':     # camera pointed at the periphery: optic disc and macula leave the field
        c = int(S * rng.uniform(0.6, 0.7)); ox, oy = rng.choice([0, S - c]), rng.integers(0, S - c)
        y = np.asarray(Image.fromarray((x[oy:oy + c, ox:ox + c] * 255).astype(np.uint8)).resize((S, S))).astype(np.float32) / 255
    return np.clip(y, 0, 1) * m

GROUPS = {'focus': ['blur', 'haze'], 'exposure': ['dark', 'washout'], 'obstruction': ['shadow', 'glare'], 'aim': ['aim']}

DINOv2 features (class token + mean patch token), computed once and cached in Drive.

In [ ]:
from transformers import AutoModel
enc = AutoModel.from_pretrained('facebook/dinov2-base').cuda().eval().half()
MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

@torch.no_grad()
def encode(xs, bs=32):
    out = []
    for i in range(0, len(xs), bs):
        b = torch.tensor((np.stack(xs[i:i + bs]) - MEAN) / STD).permute(0, 3, 1, 2).cuda().half()
        h = enc(pixel_values=b).last_hidden_state
        out.append(torch.cat([h[:, 0], h[:, 1:].mean(1)], 1).float().cpu())
    return torch.cat(out).numpy()

def simple_checks(x):
    g, m = x[..., 1], ndimage.binary_erosion(retina_mask(x), iterations=8)
    return [ndimage.laplace(g)[m].var(), g[m].mean(), g[m].std()]   # sharpness, brightness, contrast

def build(paths):
    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(load_square, paths))

cache = f'{D}/quality/features.npz'
if os.path.exists(cache):
    F = dict(np.load(cache, allow_pickle=True))
else:
    # Stream photos in chunks of 64 and keep only features, so memory stays small on any runtime
    good_paths = sorted(set(glob.glob('/content/idrid/**/*Original Images/*/*.jpg', recursive=True)))
    good_f, bad_f, bad_group = [], [], []
    for i in range(0, len(good_paths), 64):
        xs = build(good_paths[i:i + 64])
        good_f.append(encode(xs))
        for g, kinds in GROUPS.items():
            bad_f.append(encode([corrupt(x, rng.choice(kinds)) for x in xs])); bad_group += [g] * len(xs)
        if i == 0:
            examples = np.stack(xs[:3])
    dd = pd.concat([pd.read_csv(p).assign(split=s) for s, p in
                    [('train', '/content/deepdrid/regular_fundus_images/regular-fundus-training/regular-fundus-training.csv'),
                     ('val', '/content/deepdrid/regular_fundus_images/regular-fundus-validation/regular-fundus-validation.csv')]])
    dd_paths = [f"/content/deepdrid/regular_fundus_images/regular-fundus-{'training' if s == 'train' else 'validation'}/Images/{p}/{i}.jpg"
                for s, p, i in zip(dd.split, dd.patient_id, dd.image_id)]
    dd_f, dd_checks = [], []
    for i in range(0, len(dd_paths), 64):
        xs = build(dd_paths[i:i + 64])
        dd_f.append(encode(xs)); dd_checks += [simple_checks(x) for x in xs]
    F = dict(good=np.concatenate(good_f), bad=np.concatenate(bad_f), bad_group=np.array(bad_group),
             dd=np.concatenate(dd_f), dd_checks=np.array(dd_checks), dd_labels=dd.to_records(index=False))
    np.savez(cache, **F)
    np.savez(f'{D}/quality/examples.npz', good=examples)
dd = pd.DataFrame(F['dd_labels'])
print(len(F['good']), 'good |', len(F['bad']), 'synthetic bad |', len(dd), 'DeepDRiD photos,',
      f"{(dd['Overall quality'] == 0).mean():.0%} labelled not good enough")

Fit the energies (no DeepDRiD labels), then score them on DeepDRiD's real labels.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score

sc = StandardScaler().fit(np.r_[F['good'], F['bad']])
G, B, X = sc.transform(F['good']), sc.transform(F['bad']), sc.transform(F['dd'])

# 1. Contrastive: one energy per failure group (log-loss contrastive training, as in LeCun's 2006 tutorial)
E = {}
for g in GROUPS:
    Bg = B[F['bad_group'] == g]
    head = LogisticRegression(C=0.1, max_iter=2000).fit(np.r_[G, Bg], np.r_[np.zeros(len(G)), np.ones(len(Bg))])
    e_good = head.decision_function(G)
    E[g] = (head.decision_function(X) - e_good.mean()) / e_good.std()   # z-scored against good photos
contrastive = np.max(np.stack(list(E.values())), 0)
coach = np.array(list(GROUPS))[np.argmax(np.stack(list(E.values())), 0)]

# 2. Regularized: energy = Mahalanobis distance from the good photos, no negatives
lw = LedoitWolf().fit(G)
regularized = lw.mahalanobis(X)

poor = (dd['Overall quality'] == 0).values
val = (dd.split == 'val').values
chk = F['dd_checks']
scores = {'contrastive energy (max over groups)': contrastive, 'regularized energy (Mahalanobis)': regularized,
          'simple: low sharpness': -chk[:, 0], 'simple: brightness off-centre': np.abs(chk[:, 1] - np.median(chk[:, 1])),
          'simple: low contrast': -chk[:, 2]}
rows = [dict(score=k, auroc_all_1600=roc_auc_score(poor, v), auroc_val_400=roc_auc_score(poor[val], v[val])) for k, v in scores.items()]

# Supervised reference: trained on DeepDRiD's own training labels, scored on its validation split
tr = ~val
for name, feats in [('supervised ceiling: DINOv2 + logistic', X), ('supervised: simple checks + logistic', StandardScaler().fit_transform(chk))]:
    clf = LogisticRegression(C=0.1, max_iter=2000).fit(feats[tr], poor[tr])
    rows.append(dict(score=name, auroc_all_1600=np.nan, auroc_val_400=roc_auc_score(poor[val], clf.decision_function(feats[val]))))
res = pd.DataFrame(rows); res.to_csv(f'{D}/quality/results.csv', index=False)
print('AUROC for "not good enough for diagnosis" (0.5 = chance)'); print(res.round(3).to_string(index=False))

# Does each group's energy track its matching DeepDRiD sub-score? (rows: energy, columns: problem)
probs = {'clarity <= 6': dd['Clarity'] <= 6, 'field <= 6': dd['Field definition'] <= 6, 'artifact >= 6': dd['Artifact'] >= 6}
match = pd.DataFrame({p: {g: roc_auc_score(lab, E[g]) for g in GROUPS} for p, lab in probs.items()})
print('\nAUROC of each group energy for each labelled problem (focus should track clarity, aim field, obstruction artifact)')
print(match.round(3).to_string())

The demo in one plot: one good IDRiD photo, increasingly blurred. Watch the focus energy climb past the others.

In [ ]:
import matplotlib.pyplot as plt
ex = np.load(f'{D}/quality/examples.npz')['good'][0]
radii = [0, 1, 2, 3, 4, 5, 6]
xs = [ex if r == 0 else np.clip(blur(ex, r), 0, 1) * retina_mask(ex)[..., None] for r in radii]
Z = sc.transform(encode(xs))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for g in GROUPS:
    Bg = B[F['bad_group'] == g]
    head = LogisticRegression(C=0.1, max_iter=2000).fit(np.r_[G, Bg], np.r_[np.zeros(len(G)), np.ones(len(Bg))])
    e_good = head.decision_function(G)
    ax[0].plot(radii, (head.decision_function(Z) - e_good.mean()) / e_good.std(), marker='o', label=g)
ax[0].set_xlabel('blur radius (px at 448)'); ax[0].set_ylabel('energy (z vs good photos)'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].imshow(np.concatenate([xs[0], xs[3], xs[6]], 1)); ax[1].axis('off'); ax[1].set_title('blur 0 / 3 / 6')
plt.tight_layout(); plt.show()